---
**PONTIFICIA UNIVERSIDAD JAVERIANA**  
**Facultad de Ingeniería**

## Reporte Técnico: Machine Learning con PySpark
**Asignatura:** Procesamiento de Alto Volumen de Datos

---

**Estudiante:** Juan Diego Muñoz Angulo  
**Fecha de entrega:** 06/05/2026  
**Tema:** Modelos de Clasificación y Evaluación de Métricas

### 1. Resumen del Taller
Este ejercicio práctico se centra en la implementación de algoritmos de aprendizaje automático sobre un **Dataset bancario de origen portugués**. El propósito principal es evaluar la eficacia de diversos clasificadores para predecir el comportamiento de los clientes frente a la suscripción de depósitos a plazo fijo (variable objetivo $y$).

### 2. Contexto del Dataset
La información proviene de campañas de marketing directo ejecutadas mediante contacto telefónico. Debido a la naturaleza de la captación, un mismo cliente puede registrar múltiples interacciones antes de concretar o rechazar el producto financiero.

#### Estructura de los Datos Disponibles:
Para el desarrollo de las pruebas se consideran cuatro variantes del conjunto de datos original:

| Archivo | Registros | Atributos | Descripción |
| :--- | :--- | :--- | :--- |
| `bank-additional-full.csv` | 41,188 | 20 | Dataset completo ordenado cronológicamente. |
| `bank-additional.csv` | 4,119 | 20 | Muestra aleatoria del 10% del set principal. |
| `bank-full.csv` | 45,211 | 17 | Versión previa con menor número de variables. |
| `bank.csv` | 4,521 | 17 | Muestra reducida de la versión previa. |

> **Nota:** Las versiones reducidas (10%) están diseñadas específicamente para optimizar tiempos de cómputo en algoritmos de alta complejidad, como las Máquinas de Soporte Vectorial (SVM).

### 3. Objetivo de Modelado
Desarrollar un flujo de trabajo en **PySpark** que permita clasificar de forma binaria (“sí” o “no”) la probabilidad de que un usuario adquiera un depósito, basándose en el historial de las campañas de marketing.

In [ ]:
## import general-purpose libraries
import os
import sys

import numpy as np             # numerical arrays and data manipulation
import pandas as pd            # table-oriented data handling
import seaborn as sns          # statistical plots and visual styling
import matplotlib.pyplot as plt # charts and figure rendering

: 

In [ ]:
# import libraries required for Spark initialization
# findspark helps locate the installed Spark runtime
import findspark

findspark.init()

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

In [ ]:
# initialize the Spark session for distributed processing
# the configuration is basic and only includes a scheduler policy,
# a custom allocation file, a cluster master, and the application label
spark_config = SparkConf()
spark_config.set("spark.scheduler.mode", "FAIR")
spark_config.set("spark.scheduler.allocation", "/Almacen/Spark/conf/fairscheduler.xml")
spark_config.setMaster("spark://10.43.97.190:7077")
spark_config.setAppName("Taller_Banca_Metricas")

# Create the SparkSession using the configuration above jaj
spark_session = SparkSession.builder.config(conf=spark_config).getOrCreate()

spark_session

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 07:38:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Load the bank dataset into Spark and inspect the first rows
# The file uses semicolons and includes a header row.

df00 = (
    spark_session.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("bank-full.csv")
)

df00.show(5)

+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
|age|         job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+------------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+---+
| 58|  management|married| tertiary|     no|   2143|    yes|  no|unknown|  5|  may|     261|       1|   -1|       0| unknown| no|
| 44|  technician| single|secondary|     no|     29|    yes|  no|unknown|  5|  may|     151|       1|   -1|       0| unknown| no|
| 33|entrepreneur|married|secondary|     no|      2|    yes| yes|unknown|  5|  may|      76|       1|   -1|       0| unknown| no|
| 47| blue-collar|married|  unknown|     no|   1506|    yes|  no|unknown|  5|  may|      92|       1|   -1|       0| unknown| no|
| 33|     unknown| single|  unknown|     no|      1|     no|  no|unknown|  5|  may|     19

In [ ]:
# Check the dataset fields after loading
# .columns returns the list of column names from the Spark DataFrame
# We should see the expected bank dataset attributes, including the target column 'y'
print(df00.columns)

In [14]:
df00.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



In [ ]:
# Convert selected numeric fields to integer type.
# This prepares the dataset for consistent numeric analysis.

all_columns = [
    'age', 'job', 'marital', 'education',
    'default', 'balance', 'housing',
    'loan', 'contact', 'day',
    'month', 'duration', 'campaign',
    'pdays', 'previous', 'poutcome', 'y'
]

integer_columns = [
    'age',
    'balance',
    'day',
    'duration',
    'campaign',
    'pdays',
    'previous'
]

# Apply integer conversion to each numeric column.
df01 = df00.withColumn('age', df00.age.cast('int'))
df01 = df01.withColumn('balance', df01.balance.cast('int'))
df01 = df01.withColumn('day', df01.day.cast('int'))
df01 = df01.withColumn('duration', df01.duration.cast('int'))
df01 = df01.withColumn('campaign', df01.campaign.cast('int'))
df01 = df01.withColumn('pdays', df01.pdays.cast('int'))
df01 = df01.withColumn('previous', df01.previous.cast('int'))

# Confirm the type conversion by printing the schema.
df01.printSchema()

In [ ]:
# Basic target distribution check for the binary label y
# Use the cleaned DataFrame after the numeric conversions.
total_records = df01.count()
target_counts = df01.groupBy("y").count().orderBy("y")

print(f"Total records in the dataset: {total_records}")
print("Class balance for target variable y:\n")

target_distribution = target_counts.withColumn(
    "percentage",
    target_counts["count"] * 100.0 / total_records
)

# Display the binary target counts along with percentages.
target_distribution.toPandas()

In [ ]:
# Plot the target class distribution and assess imbalance risk

target_df = target_distribution.toPandas()
major_class = target_df.loc[target_df['count'].idxmax()]
minor_class = target_df.loc[target_df['count'].idxmin()]
imbalance_ratio = major_class['count'] / minor_class['count']

print(f"Majority class: {major_class['y']} -> {major_class['count']} records ({major_class['percentage']:.1f}%)")
print(f"Minority class: {minor_class['y']} -> {minor_class['count']} records ({minor_class['percentage']:.1f}%)")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio >= 2.0:
    print("Warning: the dataset is moderately imbalanced.")
else:
    print("The class distribution is relatively balanced.")

plt.figure(figsize=(8, 5))
sns.barplot(data=target_df, x='y', y='count', palette='pastel')
plt.title('Target class distribution for y')
plt.xlabel('y')
plt.ylabel('Record count')

for index, row in target_df.iterrows():
    plt.text(index, row['count'] + target_df['count'].max() * 0.01,
             f"{row['percentage']:.1f}%", ha='center')

plt.tight_layout()
plt.show()

- The target distribution is currently being evaluated for future imbalance risk, with the minority and majority classes explicitly compared.
- The plotted class counts and percentages help us understand whether the label `y` is skewed toward one class.
- The imbalance ratio is a useful early warning metric to decide if resampling or class weighting may be needed later.
- If the ratio exceeds a moderate threshold, we should monitor the dataset before training and consider balancing strategies.

In [ ]:
# Study analysis

df01.describe().toPandas().head()